# Agente de cálculo estructural (Colab)
Ejecuta las celdas **en orden** (▶). El agente calcula con OpenSees y te pide aprobación antes de emitir resultados.

> ⚠️ Resultado de ayuda al cálculo. Requiere revisión y firma de un ingeniero responsable.

**Antes de empezar** (una sola vez), en el panel 🔑 *Secretos* de Colab agrega:
- `ANTHROPIC_API_KEY` (obligatorio) y activa *Acceso desde el notebook*.
- `GITHUB_TOKEN` (solo si tu repositorio es privado).

## 1. Conectar Google Drive
Aquí se guardan la memoria de normas, el registro y las memorias de cálculo, para que no se pierdan.

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.environ['AGENTE_DIR'] = '/content/drive/MyDrive/agente_estructural'
print('Datos del agente en:', os.environ['AGENTE_DIR'])

## 2. Descargar el código e instalar dependencias

In [ ]:
import os, subprocess
from google.colab import userdata

REPO, RAMA = 'mrubilar84-ship-it/cloud', 'claude/hola-d69076'
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None

if not os.path.isdir('/content/cloud'):
    url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
    r = subprocess.run(['git', 'clone', '-b', RAMA, url, '/content/cloud'], capture_output=True, text=True)
    if r.returncode:
        raise SystemExit('No se pudo clonar (¿repositorio privado sin GITHUB_TOKEN?):\n' + r.stderr.replace(token or '#', '***'))
    subprocess.run(['git', '-C', '/content/cloud', 'remote', 'set-url', 'origin', f'https://github.com/{REPO}.git'])

%cd /content/cloud
!pip install -q -r requirements.txt anthropic

## 3. Cargar la clave de la API (desde Secretos, nunca escrita aquí)

In [ ]:
from google.colab import userdata
import anthropic

try:
    clave = userdata.get('ANTHROPIC_API_KEY').strip()
except Exception as e:
    raise SystemExit("No se encontró el secreto ANTHROPIC_API_KEY. Créalo en el panel 🔑 Secretos y activa "
                     "'Acceso desde el notebook'.")
if not clave.startswith('sk-ant-'):
    raise SystemExit("La clave no tiene el formato esperado (debe empezar con 'sk-ant-'). Recuerda: la "
                     "suscripción a Claude (claude.ai) NO incluye clave de API; créala en console.anthropic.com.")
try:
    anthropic.Anthropic(api_key=clave).models.list(limit=1)   # consulta sin costo para validar la clave
except anthropic.AuthenticationError:
    raise SystemExit("Anthropic rechazó la clave (401). Revisa que esté completa, que no esté revocada y que "
                     "venga de console.anthropic.com.")
os.environ['ANTHROPIC_API_KEY'] = clave
print('Clave válida y cargada.')

## 4. Subir tu DXF y el JSON de secciones
Selecciona **ambos archivos a la vez**. Si no tienes, escribe `demo` en la celda de abajo para usar los ejemplos.

In [ ]:
from google.colab import files

USAR_DEMO = False   # ← cámbialo a True para probar con los archivos de ejemplo

if USAR_DEMO:
    !python crear_dxf_edificio.py
    dxf, sec = 'ejemplos/edificio_2pisos.dxf', 'ejemplos/secciones_demo.json'
else:
    subidos = files.upload()
    dxf = next(n for n in subidos if n.lower().endswith('.dxf'))
    sec = next(n for n in subidos if n.lower().endswith('.json'))
print('DXF:', dxf, '| Secciones:', sec)

## 5. Ejecutar el agente
Te preguntará en la caja de texto cuando necesite datos o tu aprobación (responde `s` para aprobar). Para terminar, deja vacío el mensaje.

In [ ]:
import anthropic
from estructural.agente.agente import Agente, conversar

agente = Agente(anthropic.Anthropic(), os.environ['AGENTE_DIR'])
conversar(agente, f"Analiza esta estructura. DXF: {dxf} ; secciones: {sec}. "
                  "Propón combinaciones de carga, calcula, revisa los resultados y prepara la memoria de cálculo.")

## 6. Ver la memoria de cálculo generada

In [ ]:
import glob
from IPython.display import Markdown, display
ultimos = sorted(glob.glob(os.environ['AGENTE_DIR'] + '/salidas/*.md'), key=os.path.getmtime)
display(Markdown(open(ultimos[-1]).read())) if ultimos else print('Aún no hay memorias generadas.')